# PayFilter — Anomaly Detection & Risk Layer Evaluation

## 1. Overview
This notebook evaluates the **PayFilter Phase 1 ML Foundation** for pre-order AI transaction risk filtering.

### Temporal Validation Strategy (Time-Based Split)
**Why a random train/test split is strictly invalid for transaction risk modeling:**
In transaction fraud detection, customer behavior evolves over time, and features rely on rolling historical context (e.g., customer averages, velocity counters, and merchant shift indicators). Using a standard random train/test split causes catastrophic **lookahead data leakage**, where future transaction patterns inadvertently leak into the training distribution and artificial past data appears in the test distribution. To preserve realistic causal evaluation, we enforce a strict **time-based split**: sorting transactions chronologically, training on the earliest ~80% of days, and evaluating strictly on the unseen final ~20% of days.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Import PayFilter ML modules
from ml.features import FEATURE_COLUMNS, extract_features
from ml.baseline_rules import evaluate_baseline_rules
from ml.train_model import (
    load_secure_model,
    time_based_train_test_split,
    verify_model_integrity,
    train_isolation_forest,
)

# Optional XGBoost benchmark
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

print(f"Setup complete. XGBoost available: {XGB_AVAILABLE}")

## 2. Load Dataset & Extract Leakage-Safe Features

In [ ]:
data_path = Path("ml/data/synthetic_transactions.csv")
if not data_path.exists():
    # Fallback to root or generate
    data_path = Path("data/synthetic_transactions.csv")

print(f"Loading dataset from {data_path}...")
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Total transactions: {len(df):,}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Overall Anomaly rate: {df['label'].mean():.4%}")

print("\nExtracting leakage-safe features...")
features_df = extract_features(df)
print(f"Feature matrix shape: {features_df.shape}")

## 3. Chronological Time-Based Partitioning

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test = time_based_train_test_split(
    df, features_df, train_ratio=0.80
)

print(f"Training set: {len(X_train):,} samples (from {df_train['timestamp'].min()} to {df_train['timestamp'].max()})")
print(f"Train anomaly rate: {y_train.mean():.4%}")
print(f"Testing set:  {len(X_test):,} samples (from {df_test['timestamp'].min()} to {df_test['timestamp'].max()})")
print(f"Test anomaly rate:  {y_test.mean():.4%}")

## 4. Evaluate Models & Baselines

We evaluate 3 approaches on the exact same unseen test window:
1. **Rules-Only Heuristic Baseline** (`baseline_rules.py`)
2. **Isolation Forest Model** (`train_model.py` with tamper-verified loading)
3. **Supervised Gradient Boosting Benchmark** (XGBoost)

In [ ]:
# 1. Rules-Only Baseline
rules_preds = evaluate_baseline_rules(X_test)

# 2. Isolation Forest (Loaded via tamper-verified security layer)
model_path = Path("ml/models/isolation_forest.pkl")
meta_path = Path("ml/models/model_metadata.json")
if not model_path.exists():
    model_path = Path("models/isolation_forest.pkl")
    meta_path = Path("models/model_metadata.json")

iso_model, metadata = load_secure_model(model_path, meta_path)
iso_raw_preds = iso_model.predict(X_test[FEATURE_COLUMNS])
iso_preds = np.where(iso_raw_preds == -1, 1, 0)

# 3. XGBoost Benchmark (Supervised, trained on X_train)
if XGB_AVAILABLE:
    xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss')
    xgb_clf.fit(X_train[FEATURE_COLUMNS], y_train)
    xgb_preds = xgb_clf.predict(X_test[FEATURE_COLUMNS])
else:
    xgb_preds = np.zeros(len(y_test), dtype=int)

print("Inference complete for all models.")

## 5. False Positive & Friction Cost Analysis

### Friction Cost Formula
When a legitimate transaction is incorrectly flagged (False Positive), user friction (verification step, OTP challenge, or hold) causes a percentage of genuine users to drop off.

$$\text{Friction Cost} = \text{False Positives} \times \text{Average Transaction Value} \times \text{Assumed Drop-off Rate}$$

> **Stated Assumption:** We assume an industry-standard **5% drop-off rate** from verification friction (this is an assumption for comparative cost modeling, not an empirical ground truth).

In [ ]:
ASSUMED_DROP_OFF_RATE = 0.05  # 5% assumed customer drop-off on friction
avg_txn_value = float(df_test['amount'].mean())
num_legit = int((y_test == 0).sum())

def calculate_model_metrics(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    fp_percentage = (fp / num_legit) * 100.0 if num_legit > 0 else 0.0
    friction_cost = fp * avg_txn_value * ASSUMED_DROP_OFF_RATE
    
    return {
        "Model": name,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1 Score": round(f1, 4),
        "False Positive Rate": f"{fpr:.2%}",
        "Legitimate Flagged (FP)": int(fp),
        "Legit Flagged (%)": f"{fp_percentage:.2f}%",
        "Est. Friction Cost (INR)": f"₹{friction_cost:,.2f}",
    }

metrics_list = [
    calculate_model_metrics("Rules-Only Baseline", y_test.values, rules_preds),
    calculate_model_metrics("Isolation Forest (Unsupervised)", y_test.values, iso_preds),
]

if XGB_AVAILABLE:
    metrics_list.append(calculate_model_metrics("XGBoost (Supervised Benchmark)", y_test.values, xgb_preds))

results_df = pd.DataFrame(metrics_list)
results_df

## 6. Confusion Matrices & Detailed Breakdown

In [ ]:
for name, preds in [("Rules Baseline", rules_preds), ("Isolation Forest", iso_preds)]:
    print(f"\n{'='*20} {name} Classification Report {'='*20}")
    print(classification_report(y_test, preds, target_names=['Normal (0)', 'Anomalous (1)'], digits=4))
    print("Confusion Matrix (TN, FP / FN, TP):")
    print(confusion_matrix(y_test, preds))